In [1]:
import sys
sys.path.append("../")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LightSource
from sklearn.preprocessing import QuantileTransformer
import skgstat as skg
from skgstat import models
import gstatsim_torch as gst
import parallel_torch as gspt
import torch
import math

In [2]:
from torch.profiler import profile, record_function, ProfilerActivity

## Data Preparation

In [3]:
df_bed = pd.read_csv('demos/data/greenland_test_data.csv')

# remove erroneously high values due to bad bed picks
df_bed = df_bed[df_bed["Bed"] <= 700]  

In [4]:
# grid data to 100 m resolution and remove coordinates with NaNs
res = 1000
df_grid, torch_data, rows, cols = gst.Gridding.grid_data(df_bed, 'X', 'Y', 'Bed', res)
df_grid = df_grid[df_grid["Z"].isnull() == False]
torch_data = torch_data[torch_data[:,2].isnan() == False]
df_grid = df_grid.rename(columns = {"Z": "Bed"})

# normal score transformation
data = df_grid['Bed'].values.reshape(-1,1)
nst_trans = QuantileTransformer(n_quantiles=500, output_distribution="normal").fit(data)
df_grid['Nbed'] = nst_trans.transform(data)

# compute experimental (isotropic) variogram
coords = df_grid[['X','Y']].values
values = df_grid['Nbed']
torch_data[:,2] = torch.tensor(df_grid['Nbed'].to_numpy())

maxlag = 50000             # maximum range distance
n_lags = 70                # num of bins

V1 = skg.Variogram(coords, values, bin_func='even', n_lags=n_lags, 
                   maxlag=maxlag, normalize=False)

# use exponential variogram model
V1.model = 'exponential'
V1.parameters

[np.float64(31852.500832412148), np.float64(0.7027444989852435), 0]

In [5]:
# define coordinate grid
xmin = torch.min(torch_data[:,0]); xmax = torch.max(torch_data[:,0])     # min and max x values
ymin = torch.min(torch_data[:,1]); ymax = torch.max(torch_data[:,1])     # min and max y values

Pred_grid_xy = gst.Gridding.prediction_grid(xmin, xmax, ymin, ymax, res)

In [6]:
# set variogram parameters
azimuth = 0
nugget = V1.parameters[2]

# the major and minor ranges are the same in this example because it is isotropic
major_range = V1.parameters[0]
minor_range = V1.parameters[0]
sill = V1.parameters[1]
vtype = 'Exponential'

# save variogram parameters as a list
vario = [azimuth, nugget, major_range, minor_range, sill, vtype]


k = 48         # number of neighboring data points used to estimate a given point
rad = 50000     # 50 km search radius

## Vectorized and Parallel torch scheme
Not actually parallelized with starmap but set up so that it can easily be converted

In [15]:
# convert parameters to simulation function naming scheme
prediction_grid = Pred_grid_xy
torch_data = torch_data
num_points = k
vario = vario
radius = rad
num_gpus = 3 * 4# more than actual to not run into memory constraints (make batch tensor smaller)

In [16]:
# setup before parallel function

observed_coords = torch_data[:,:2].tolist()
simulate_coords = [coord for coord in prediction_grid.tolist() if coord not in observed_coords]

observed_coords = torch.tensor(observed_coords)
simulate_coords = torch.tensor(simulate_coords)

# Shuffle data to predict to create a random path
index = torch.arange(len(simulate_coords)) 
shuffle = index[torch.randperm(len(simulate_coords))]
simulate_coords = simulate_coords[shuffle]

full = torch.vstack((observed_coords, simulate_coords))

azimuth = vario[0]
major_range = vario[2]
minor_range = vario[3]

rotation_matrix = gst.make_rotation_matrix(azimuth, major_range, minor_range, "cpu")

# create starting index for data from full to use for KNN
begin = len(observed_coords)

num_cells = len(simulate_coords)
cells_per_process = num_cells//num_gpus

i_list = [[i for i in range(j*cells_per_process, (j+1)*cells_per_process)] for j in range(num_gpus-1)]
i_list.append([i for i in range((num_gpus-1)*cells_per_process,num_cells)])

gpu_num = [i for i in range(num_gpus)]

In [17]:
# convert parameters to parallel function naming scheme
i_list = i_list[0]
gpu_id = gpu_num[0]
full = full
vario = vario
radius = radius
num_points = num_points
begin = begin
rotation_matrix = rotation_matrix

In [18]:
# Assign the GPU
torch.cuda.set_device(gpu_id)

# send data to GPU 
full = full.cuda()
rotation_matrix = rotation_matrix.cuda()

In [19]:
# BEGIN VECTORIZED CODE
offset = torch.tensor([i+begin for i in i_list]).cuda()

loc = full[offset]

search_candidates = torch.full((len(i_list), len(full), 2), float('nan')).cuda()
for i, cur_offset in enumerate(offset):
    search_candidates[i, :cur_offset] = full[:cur_offset]

## NNS

In [20]:
# RENAME FOR NNS 
data2 = search_candidates

In [21]:
locx = loc[:, 0].unsqueeze(1).repeat(1, data2.shape[1])
locy = loc[:, 1].unsqueeze(1).repeat(1, data2.shape[1])

x_tensor = data2[:, :, 0]
y_tensor = data2[:, :, 1]

centered_x = x_tensor - locx
centered_y = y_tensor - locy

distances = torch.sqrt(centered_x**2 + centered_y**2)
angles = torch.atan2(centered_y, centered_x)

# Stack the tensors into a single tensor
stack = torch.stack((x_tensor, y_tensor, distances, angles), dim=2)

indicies = torch.arange(data2.shape[1]).unsqueeze(0).repeat(data2.shape[0],1).cuda()

In [22]:
# Filter out points outside the radius
mask = torch.where((stack[:, :, 2] < radius), 1.0, float('nan')).cuda() # The distances are at index 3
stack = stack * mask.unsqueeze(2).repeat(1, 1, 4)
indicies = indicies * mask

In [23]:
# Sort stack and indicies by distance
sorted_dist_idxs = torch.argsort(stack[..., 2]).reshape(-1)
stack_idxs = (torch.arange(stack.shape[0]).repeat_interleave(stack.shape[1]).reshape(-1))
stack = stack[stack_idxs, sorted_dist_idxs, :].reshape(*stack.shape)
indicies = indicies[stack_idxs, sorted_dist_idxs].reshape(*indicies.shape)

In [24]:
# Use bucketize to find bin index for each angle
bins = torch.tensor([-math.pi, -3*math.pi/4, -math.pi/2, -math.pi/4, 0,
                        math.pi/4, math.pi/2, 3*math.pi/4, math.pi]).cuda()
bin_indices = torch.bucketize(stack[..., 3].contiguous(), bins, right=False)  # The angles are at index 3

In [25]:
# Allocate tensor for the result
smallest = torch.full((len(i_list), num_points, 2), float('nan')).cuda()
index_list = torch.full((len(i_list), num_points), float('nan')).cuda()
oct_count = num_points // 8

In [27]:
# Remember cant clean up NAN in smallest or index_list because nonequal number of NAN for each row 

for i in range(1, bins.shape[0]):
    bin_mask = torch.where((bin_indices == i), 1.0, float('nan')).cuda()
    bin_points = (stack * bin_mask.unsqueeze(2).repeat(1, 1, 4))[..., :2]
    bin_index = indicies * bin_mask
    bin_points_count = torch.minimum(torch.tensor([oct_count]).cuda(), torch.sum(~torch.isnan(bin_index), dim=1))
    
    for j, count in enumerate(bin_points_count): 
        if count > 0:

            cur_loc_NN = bin_points[j]
            cur_loc_idx_list = bin_index[j]

            smallest[j, (i-1) * oct_count : (i-1) * oct_count + count, :] = cur_loc_NN[~torch.isnan(cur_loc_NN[:, 0])][:count]
            index_list[j, (i-1) * oct_count : (i-1) * oct_count + count] = cur_loc_idx_list[~torch.isnan(cur_loc_idx_list)][:count]


## SKRIG

In [40]:
for i, near in enumerate(smallest):
    if i == 0:
        print(near)
        print(index_list[i])
        print(near[~torch.isnan(near[:, 0])].reshape(-1,2))
        print(index_list[i][~torch.isnan(index_list[i])])

SyntaxError: incomplete input (3190899487.py, line 6)